In [3]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy import ndimage

#######################################################
# CONFIGURAÇÕES
#######################################################

# Caminhos dos CSVs
CIDADES = {
    'londrina': '/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_londrina/inference_results_with_positions_20250609_083252.csv',
    'ibipora': '/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_ibipora/inference_results_with_positions_20250609_164529.csv',
    'cambe': '/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_cambe/inference_results_with_positions_20250609_161440.csv',
    'apucarana': '/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_apucarana/inference_results_with_positions_20251118_114239.csv',
    'arapongas': '/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_arapongas/inference_results_with_positions_20251118_121155.csv',
    'cambira': '/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_cambira/inference_results_with_positions_20251118_122253.csv',
    'jandaia': '/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_jandaia/inference_results_with_positions_20251118_122910.csv',
    'mandaguari': '/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_mandaguari/inference_results_with_positions_20251118_125143.csv',
    'marialva': '/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_marialva/inference_results_with_positions_20251118_124329.csv',
    'rolandia': '/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_rolandia/inference_results_with_positions_20251118_130921.csv',
    'sarandi': '/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_sarandi/inference_results_with_positions_20251118_131430.csv',
    'maringa': '/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_maringa/inference_results_with_positions_20251118_133121.csv'
}


# Mapeamentos de agregação
MAPEAMENTO_BUILT_UP = {
    'hduf': 1, 'lduf': 1, 'mduf': 1, 'industrial': 1,
    'crop': 2, 'bush': 2, 'tree': 2, 'bare': 2, 'water': 2, 'grass': 2
}

MAPEAMENTO_INDUSTRIAL = {
    'industrial': 1,
    'hduf': 2, 'lduf': 2, 'mduf': 2,
    'crop': 2, 'bush': 2, 'tree': 2, 'bare': 2, 'water': 2, 'grass': 2
}



CLASSE_NOMES = {
    'hduf': 'Alta Densidade Urbana',
    'lduf': 'Baixa Densidade Urbana',
    'mduf': 'Média Densidade Urbana',
    'industrial': 'Industrial',
    'crop': 'Cultivo',
    'bush': 'Arbustivo',
    'tree': 'Floresta',
    'bare': 'Solo Exposto',
    'water': 'Água',
    'grass': 'Gramíneas'
}

TAMANHO_PATCH = 109.45  # metros

#######################################################
# FUNÇÕES AUXILIARES
#######################################################

def parse_coords(s):
    """Extrai lat, lon de string 'lat,lon'"""
    lat, lon = map(float, str(s).split(','))
    return lat, lon


def carregar_cidade(cidade):
    """Carrega o CSV da cidade"""
    if cidade not in CIDADES:
        raise ValueError(f"Cidade '{cidade}' não encontrada. Opções: {list(CIDADES.keys())}")
    
    # Tentar detectar separador
    df = pd.read_csv(CIDADES[cidade], sep=None, engine='python')
    
    # Se não funcionou, tentar com ponto e vírgula
    if len(df.columns) == 1:
        df = pd.read_csv(CIDADES[cidade], sep=';')
    
    print(f"✓ {cidade.upper()}: {len(df):,} patches carregados")
    print(f"  Colunas: {df.columns.tolist()[:5]}...")  # mostrar primeiras 5
    return df


def preparar_dados(df):
    """Prepara coordenadas do DataFrame - detecta automaticamente as colunas"""
    df = df.copy()
    
    colunas = df.columns.tolist()
    print(f"Colunas detectadas: {colunas}")
    
    if 'center' in colunas:
        coords = df['center'].apply(parse_coords)
        df['latitude'] = coords.apply(lambda x: x[0])
        df['longitude'] = coords.apply(lambda x: x[1])
        
    elif 'top_left' in colunas and 'bottom_right' in colunas:
        tl = df['top_left'].apply(parse_coords)
        br = df['bottom_right'].apply(parse_coords)
        df['latitude'] = (tl.apply(lambda x: x[0]) + br.apply(lambda x: x[0])) / 2
        df['longitude'] = (tl.apply(lambda x: x[1]) + br.apply(lambda x: x[1])) / 2
        
    elif 'latitude' in colunas and 'longitude' in colunas:
        pass  # já tem as colunas
        
    elif 'lat' in colunas and 'lon' in colunas:
        df['latitude'] = df['lat']
        df['longitude'] = df['lon']
        
    else:
        raise ValueError(f"Colunas de coordenadas não encontradas. Disponíveis: {colunas}")
    
    print(f"✓ Coordenadas preparadas: lat [{df['latitude'].min():.4f}, {df['latitude'].max():.4f}], lon [{df['longitude'].min():.4f}, {df['longitude'].max():.4f}]")
    
    return df

def agregar_classes(df, tipo_agregacao='built_up'):
    """Agrega classes conforme tipo escolhido."""
    df = df.copy()
    
    if tipo_agregacao == 'built_up':
        mapeamento = MAPEAMENTO_BUILT_UP
        nome_classe = 'Built Up'
    elif tipo_agregacao == 'industrial':
        mapeamento = MAPEAMENTO_INDUSTRIAL
        nome_classe = 'Industrial'
    else:
        raise ValueError(f"tipo_agregacao deve ser 'built_up' ou 'industrial'")
    
    df['classe_agregada'] = df['predicted_class'].map(mapeamento)
    
    nao_mapeadas = df[df['classe_agregada'].isna()]
    if len(nao_mapeadas) > 0:
        classes_problema = nao_mapeadas['predicted_class'].unique()
        print(f"⚠️ {len(nao_mapeadas)} patches não mapeados: {classes_problema}")
        df = df.dropna(subset=['classe_agregada'])
    
    df['classe_agregada'] = df['classe_agregada'].astype(int)
    
    n_classe1 = len(df[df['classe_agregada'] == 1])
    n_classe2 = len(df[df['classe_agregada'] == 2])  # Mudou de 0 para 2
    
    print(f"✓ {nome_classe}: {n_classe1:,} patches")
    print(f"✓ Non-{nome_classe}: {n_classe2:,} patches")
    
    return df


def criar_raster(df, cell_size=TAMANHO_PATCH):
    """Cria raster a partir do DataFrame usando coordenadas métricas"""
    
    # Criar GeoDataFrame
    gdf = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df['longitude'], df['latitude']),
        crs="EPSG:4326"
    )
    
    # Reprojetar para coordenadas métricas
    gdf_utm = gdf.to_crs("EPSG:29192")
    gdf_utm['x_utm'] = gdf_utm.geometry.x
    gdf_utm['y_utm'] = gdf_utm.geometry.y
    
    # Criar índices de grade
    x0 = gdf_utm['x_utm'].min()
    y_max = gdf_utm['y_utm'].max()
    
    gdf_utm['col_index'] = ((gdf_utm['x_utm'] - x0) / cell_size).round().astype(int)
    gdf_utm['row_index'] = ((y_max - gdf_utm['y_utm']) / cell_size).round().astype(int)
    
    # Criar raster
    n_rows = gdf_utm['row_index'].max() + 1
    n_cols = gdf_utm['col_index'].max() + 1
    
    raster = np.zeros((n_rows, n_cols), dtype=np.int32)
    
    for _, row in gdf_utm.iterrows():
        r = int(row['row_index'])
        c = int(row['col_index'])
        if 0 <= r < n_rows and 0 <= c < n_cols:
            raster[r, c] = row['classe_agregada']
    
    print(f"✓ Raster criado: {raster.shape}")
    
    return raster, gdf_utm

#######################################################
# FUNÇÕES DE MÉTRICAS
#######################################################

def calcular_shannon(raster_original):
    """
    Calcula Shannon Index para as 10 classes originais.
    Precisa do raster com classes 1-10, não agregado.
    """
    values = raster_original[raster_original > 0]
    unique, counts = np.unique(values, return_counts=True)
    proportions = counts / counts.sum()
    shannon = -np.sum(proportions * np.log(proportions))
    return shannon


def calcular_metricas_paisagem(raster, pixel_size_m=TAMANHO_PATCH, classe_alvo=1):
    """
    Calcula todas as métricas de paisagem para a classe alvo.
    
    Métricas calculadas:
    - Área total (km²)
    - Perímetro total / Edge length (km)
    - Edge density (km/km²)
    - Number of patches (4-conn e 8-conn)
    - Patch density
    - Mean patch area (km²)
    - Median patch area (km²)
    - Greatest patch area (km²)
    - Smallest patch area (km²)
    - Largest Patch Index (%)
    - Mean patch shape ratio
    - Patch cohesion index
    - Landscape proportion (%)
    """
    
       # Máscara da classe alvo
    classe_mask = (raster == classe_alvo).astype(int)
    
    # Estruturas de conectividade
    structure_4 = np.array([[0,1,0], [1,1,1], [0,1,0]])
    structure_8 = np.ones((3,3))
    
    # --- Métricas básicas ---
    total_pixels = np.sum(raster > 0)  # todos os pixels com dados (1 ou 2)
    classe_pixels = np.sum(classe_mask)

    
    area_m2 = classe_pixels * (pixel_size_m ** 2)
    area_km2 = area_m2 / 1e6
    
    area_total_m2 = total_pixels * (pixel_size_m ** 2)
    area_total_km2 = area_total_m2 / 1e6
    
    # Landscape proportion
    landscape_proportion = (classe_pixels / total_pixels) * 100 if total_pixels > 0 else 0
    
    # --- Edge length e Edge density ---
    h_edges = np.sum(np.abs(np.diff(classe_mask, axis=1)))
    v_edges = np.sum(np.abs(np.diff(classe_mask, axis=0)))
    total_edges = h_edges + v_edges
    
    edge_length_m = total_edges * pixel_size_m
    edge_length_km = edge_length_m / 1000
    
    edge_density = edge_length_km / area_km2 if area_km2 > 0 else 0
    
    # --- Number of Patches ---
    labeled_4, num_patches_4 = ndimage.label(classe_mask, structure=structure_4)
    labeled_8, num_patches_8 = ndimage.label(classe_mask, structure=structure_8)
    
    # Usar 8-conectividade para estatísticas de patches (padrão LecoS/FRAGSTATS)
    labeled = labeled_8
    num_patches = num_patches_8
    
    # Patch density (patches por km²)
    patch_density = num_patches / area_total_km2 if area_total_km2 > 0 else 0
    
    # --- Patch statistics ---
    patch_areas_px = []
    patch_perimeters_px = []
    
    for i in range(1, num_patches + 1):
        patch = (labeled == i)
        area_px = np.sum(patch)
        
        # Perímetro do patch
        h_e = np.sum(np.abs(np.diff(patch.astype(int), axis=1)))
        v_e = np.sum(np.abs(np.diff(patch.astype(int), axis=0)))
        perim_px = h_e + v_e
        
        patch_areas_px.append(area_px)
        patch_perimeters_px.append(perim_px)
    
    patch_areas_px = np.array(patch_areas_px)
    patch_perimeters_px = np.array(patch_perimeters_px)
    
    # Converter para km²
    patch_areas_km2 = patch_areas_px * (pixel_size_m ** 2) / 1e6
    patch_perimeters_km = patch_perimeters_px * pixel_size_m / 1000
    
    # Estatísticas de área dos patches
    mean_patch_area_km2 = np.mean(patch_areas_km2) if len(patch_areas_km2) > 0 else 0
    median_patch_area_km2 = np.median(patch_areas_km2) if len(patch_areas_km2) > 0 else 0
    greatest_patch_area_km2 = np.max(patch_areas_km2) if len(patch_areas_km2) > 0 else 0
    smallest_patch_area_km2 = np.min(patch_areas_km2) if len(patch_areas_km2) > 0 else 0
    
    # Largest Patch Index (%)
    lpi = (np.max(patch_areas_px) / total_pixels) * 100 if total_pixels > 0 and len(patch_areas_px) > 0 else 0
    
    # Mean Patch Shape Ratio (normalizado pelo círculo)
    if len(patch_areas_px) > 0:
        shape_ratios = patch_perimeters_px / (2 * np.sqrt(np.pi * patch_areas_px))
        mean_shape_ratio = np.mean(shape_ratios)
    else:
        mean_shape_ratio = 0
    
    # --- Patch Cohesion ---
    if len(patch_perimeters_px) > 0 and total_pixels > 1:
        sum_p = np.sum(patch_perimeters_px)
        sum_p_sqrt_a = np.sum(patch_perimeters_px * np.sqrt(patch_areas_px))
        
        if sum_p_sqrt_a > 0:
            cohesion = (1 - (sum_p / sum_p_sqrt_a)) * (1 - (1 / np.sqrt(total_pixels))) ** -1 * 100
        else:
            cohesion = 0
    else:
        cohesion = 0
    
    # --- P/A ratio (total) ---
    pa_ratio = edge_length_km / area_km2 if area_km2 > 0 else 0
    
    return {
        'area_km2': area_km2,
        'perimetro_km': edge_length_km,
        'pa_ratio': pa_ratio,
        'edge_density': edge_density,
        'num_patches_4conn': num_patches_4,
        'num_patches_8conn': num_patches_8,
        'patch_density': patch_density,
        'mean_patch_area_km2': mean_patch_area_km2,
        'median_patch_area_km2': median_patch_area_km2,
        'greatest_patch_area_km2': greatest_patch_area_km2,
        'smallest_patch_area_km2': smallest_patch_area_km2,
        'largest_patch_index': lpi,
        'mean_shape_ratio': mean_shape_ratio,
        'patch_cohesion': cohesion,
        'landscape_proportion': landscape_proportion,
        'area_total_km2': area_total_km2
    }

def criar_raster_10classes(df, cell_size=TAMANHO_PATCH):
    """Cria raster com as 10 classes originais para cálculo do Shannon"""
    
    mapeamento_10classes = {
        'bare': 1, 'bush': 2, 'crop': 3, 'grass': 4,
        'hduf': 5, 'industrial': 6, 'lduf': 7, 'mduf': 8,
        'tree': 9, 'water': 10
    }
    
    # Preparar coordenadas se ainda não tiver
    if 'latitude' not in df.columns:
        df = preparar_dados(df)
    
    gdf = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df['longitude'], df['latitude']),
        crs="EPSG:4326"
    )
    
    gdf_utm = gdf.to_crs("EPSG:29192")
    gdf_utm['x_utm'] = gdf_utm.geometry.x
    gdf_utm['y_utm'] = gdf_utm.geometry.y
    
    x0 = gdf_utm['x_utm'].min()
    y_max = gdf_utm['y_utm'].max()
    
    gdf_utm['col_index'] = ((gdf_utm['x_utm'] - x0) / cell_size).round().astype(int)
    gdf_utm['row_index'] = ((y_max - gdf_utm['y_utm']) / cell_size).round().astype(int)
    
    n_rows = gdf_utm['row_index'].max() + 1
    n_cols = gdf_utm['col_index'].max() + 1
    
    raster = np.zeros((n_rows, n_cols), dtype=np.int32)
    
    for _, row in gdf_utm.iterrows():
        r = int(row['row_index'])
        c = int(row['col_index'])
        classe_val = mapeamento_10classes.get(row['predicted_class'], 0)
        if 0 <= r < n_rows and 0 <= c < n_cols:
            raster[r, c] = classe_val
    
    return raster


#######################################################
# FUNÇÃO PRINCIPAL
#######################################################

def calcular_todas_metricas(cidade, tipo_agregacao='built_up'):
    """
    Calcula todas as métricas de paisagem para uma cidade.
    
    Parâmetros:
    -----------
    cidade : str
        Nome da cidade (ex: 'londrina', 'maringa')
    tipo_agregacao : str
        'built_up' - hduf, lduf, mduf, industrial vs resto
        'industrial' - industrial vs resto
    
    Retorna:
    --------
    dict com todas as métricas
    """
    
    print("=" * 60)
    print(f"MÉTRICAS DE PAISAGEM - {cidade.upper()}")
    print(f"Agregação: {tipo_agregacao}")
    print("=" * 60)
    
    # 1. Carregar dados
    df = carregar_cidade(cidade)
    df = preparar_dados(df)
    
    # 2. Calcular Shannon (10 classes originais)
    print("\nCalculando Shannon Index (10 classes)...")
    raster_10 = criar_raster_10classes(df)
    shannon = calcular_shannon(raster_10)
    print(f"✓ Shannon Index: {shannon:.4f}")
    
    # 3. Agregar classes
    print(f"\nAgregando classes ({tipo_agregacao})...")
    df_agregado = agregar_classes(df, tipo_agregacao)
    
    # 4. Criar raster binário
    print("\nCriando raster...")
    raster, gdf_utm = criar_raster(df_agregado)
    
    # 5. Calcular métricas
    print("\nCalculando métricas de paisagem...")
    metricas = calcular_metricas_paisagem(raster, classe_alvo=1)
    
    # 6. Adicionar informações gerais
    resultados = {
        'cidade': cidade,
        'tipo_agregacao': tipo_agregacao,
        'shannon_index': shannon,
        **metricas
    }
    
    # 7. Exibir resultados
    print("\n" + "=" * 60)
    print("RESULTADOS")
    print("=" * 60)
    print(f"\nÍndice de Shannon (SHDI): {shannon:.4f}")
    print(f"\nÁrea da classe alvo: {metricas['area_km2']:.2f} km²")
    print(f"Área total da paisagem: {metricas['area_total_km2']:.2f} km²")
    print(f"Proporção da paisagem: {metricas['landscape_proportion']:.2f}%")
    print(f"\nPerímetro total: {metricas['perimetro_km']:.2f} km")
    print(f"Razão P/A: {metricas['pa_ratio']:.4f} km/km²")
    print(f"Edge density: {metricas['edge_density']:.4f} km/km²")
    print(f"\nNúmero de patches (4-conn): {metricas['num_patches_4conn']}")
    print(f"Número de patches (8-conn): {metricas['num_patches_8conn']}")
    print(f"Patch density: {metricas['patch_density']:.4f} patches/km²")
    print(f"\nMean patch area: {metricas['mean_patch_area_km2']:.4f} km²")
    print(f"Median patch area: {metricas['median_patch_area_km2']:.4f} km²")
    print(f"Greatest patch area: {metricas['greatest_patch_area_km2']:.4f} km²")
    print(f"Smallest patch area: {metricas['smallest_patch_area_km2']:.6f} km²")
    print(f"\nLargest Patch Index: {metricas['largest_patch_index']:.2f}%")
    print(f"Mean Shape Ratio: {metricas['mean_shape_ratio']:.4f}")
    print(f"Patch Cohesion: {metricas['patch_cohesion']:.4f}")
    print("=" * 60)
    
    return resultados

def calcular_todas_cidades(tipo_agregacao='built_up'):
    """
    Calcula métricas para todas as cidades e retorna DataFrame.
    """
    resultados = []
    
    for cidade in CIDADES.keys():
        try:
            r = calcular_todas_metricas(cidade, tipo_agregacao)
            resultados.append(r)
            print("\n")
        except Exception as e:
            print(f"⚠️ Erro em {cidade}: {e}")
    
    df_resultados = pd.DataFrame(resultados)
    return df_resultados

#######################################################
# USO
#######################################################

# if __name__ == "__main__":
    
#     # --- Para uma cidade específica ---
#     # resultados = calcular_todas_metricas('londrina', tipo_agregacao='built_up')
#     # resultados = calcular_todas_metricas('londrina', tipo_agregacao='industrial')
    
#     # --- Para todas as cidades ---
#     # df_built = calcular_todas_cidades(tipo_agregacao='built_up')
#     # df_built.to_csv('metricas_built_up.csv', index=False)
    
#     # df_indust = calcular_todas_cidades(tipo_agregacao='industrial')
#     # df_indust.to_csv('metricas_industrial.csv', index=False)
    
#     # Exemplo de uso:
#     resultados = calcular_todas_metricas('londrina', tipo_agregacao='built_up')

In [4]:
# Rodar para todas as cidades
df_resultados = calcular_todas_cidades(tipo_agregacao='built_up')
df_resultados.to_csv('/Users/fjcosta/Documents/landCoverlandValue/fragment/indices/metricas_built_up_todas_cidades.csv', index=False)

# Ou para industrial
df_industrial = calcular_todas_cidades(tipo_agregacao='industrial')
df_industrial.to_csv('/Users/fjcosta/Documents/landCoverlandValue/fragment/indices/metricas_industrial_todas_cidades.csv', index=False)

MÉTRICAS DE PAISAGEM - LONDRINA
Agregação: built_up
✓ LONDRINA: 18,302 patches carregados
  Colunas: ['name', 'top_left', 'top_right', 'bottom_left', 'bottom_right']...
Colunas detectadas: ['name', 'top_left', 'top_right', 'bottom_left', 'bottom_right', 'center', 'label', 'image_path', 'predicted_class', 'prediction_confidence', 'prob_bare', 'prob_bush', 'prob_crop', 'prob_grass', 'prob_hduf', 'prob_industrial', 'prob_lduf', 'prob_mduf', 'prob_tree', 'prob_water']
✓ Coordenadas preparadas: lat [-23.4401, -23.2352], lon [-51.2480, -51.0806]

Calculando Shannon Index (10 classes)...
✓ Shannon Index: 1.8586

Agregando classes (built_up)...
✓ Built Up: 8,938 patches
✓ Non-Built Up: 9,364 patches

Criando raster...
✓ Raster criado: (208, 157)

Calculando métricas de paisagem...

RESULTADOS

Índice de Shannon (SHDI): 1.8586

Área da classe alvo: 106.99 km²
Área total da paisagem: 218.86 km²
Proporção da paisagem: 48.88%

Perímetro total: 513.54 km
Razão P/A: 4.8000 km/km²
Edge density: 4.800